# Ablação — BCE ponderada × BCE simples (60 épocas)

Notebook **só para testes**: pré-processa uma vez e treina dois modelos em sequência,
com tudo igual exceto a perda:

- **A — BCE ponderada** (`pos_weight`, cenário atual da proposta);
- **B — BCE simples** (`--no-pos-weight`, como no código de Zhang et al., 2021).

Não gera SHAP nem métricas de explicação. Ativar **GPU** e **Internet** e anexar o
dataset `physionet/china-physiological-signal-challenge-in-2018`.
Tempo esperado: até ~2 × 60 × 105 s ≈ 3,5 h na T4 (menos se o early stopping parar antes).


In [ ]:
# Célula 1 — clonar / atualizar o repositório
import os

REPO_URL = 'https://github.com/Arthur-so/shap-ecg-arrhythmia.git'
REPO_DIR = '/kaggle/working/shap-ecg-arrhythmia'

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -1


In [ ]:
# Célula 2 — instalar dependências (torch já vem no Kaggle)
!pip install -q -r requirements.txt


In [ ]:
# Célula 3 — caminhos e parâmetros
import glob, torch

RAW_DIR = '/kaggle/input'
PROC_DIR = '/kaggle/working/processed'
OUT_DIR = '/kaggle/working/ablation'

EPOCHS = 60
LR = 1e-4
BATCH_SIZE = 32
PATIENCE = 10
DROPOUT = 0.2

EXPERIMENTS = {
    'A_bce_ponderada': '',
    'B_bce_simples': '--no-pos-weight',
}

print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('nº de arquivos .hea:', len(glob.glob(RAW_DIR + '/**/*.hea', recursive=True)))


In [ ]:
# Célula 4 — pré-processamento (uma vez para os dois experimentos)
!python -m src.data.preprocess --raw-dir {RAW_DIR} --out-dir {PROC_DIR} --skip-if-exists


In [ ]:
# Célula 5 — treino dos dois experimentos, em sequência
import os
for name, extra in EXPERIMENTS.items():
    exp_dir = f'{OUT_DIR}/{name}'
    print(f'\n========== {name} ==========')
    !python -m src.train \
        --data-dir {PROC_DIR} \
        --out-dir {exp_dir}/checkpoints \
        --results-dir {exp_dir}/results \
        --epochs {EPOCHS} --lr {LR} --batch-size {BATCH_SIZE} \
        --patience {PATIENCE} --dropout {DROPOUT} {extra}
    if not os.path.exists(f'{exp_dir}/results/f1_scores.json'):
        raise RuntimeError(f'treino {name} falhou: f1_scores.json não foi gerado')


In [ ]:
# Célula 6 — comparação
import json
import pandas as pd

res = {name: json.load(open(f'{OUT_DIR}/{name}/results/f1_scores.json')) for name in EXPERIMENTS}
rows = []
for name, r in res.items():
    rows.append({'experimento': name, 'melhor época': r['best_epoch'],
                 'val (limiares)': r['val_f1']['macro'],
                 'teste (limiares)': r['test_f1']['macro'],
                 'teste (0,5)': r['test_f1_limiar_05']['macro']})
print(pd.DataFrame(rows).round(4).to_string(index=False))

classes = [c for c in res['A_bce_ponderada']['test_f1'] if c != 'macro']
print('\nF1 por classe no teste (limiares por classe):')
print(pd.DataFrame({n: {c: r['test_f1'][c] for c in classes} for n, r in res.items()}).round(3).to_string())
print('\nLimiares escolhidos na validação:')
print(pd.DataFrame({n: r['thresholds'] for n, r in res.items()}).round(2).to_string())
